### InMemoryVectorStore
In-memory vector store implementation.

Uses a dictionary, and computes cosine similarity for search using numpy.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_core.embeddings import Embeddings
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from sklearn.feature_extraction.text import HashingVectorizer

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY:
    from langchain.chat_models import init_chat_model
    llm = init_chat_model("openai:gpt-4o-mini")
    print("Using OpenAI chat model from the API.")
else:
    llm = FakeListChatModel(
        responses=["Demo answer generated without an external LLM. Add OPENAI_API_KEY for live responses."]
    )
    print("OPENAI_API_KEY not found. Using FakeListChatModel so the notebook still runs.")

class HashEmbeddings(Embeddings):
    """Deterministic local fallback embeddings for offline notebook runs."""

    def __init__(self, n_features=1536):
        self.vectorizer = HashingVectorizer(
            n_features=n_features,
            alternate_sign=False,
            norm="l2",
            ngram_range=(1, 2),
        )

    def _embed(self, texts):
        return self.vectorizer.transform(texts).toarray().astype(float).tolist()

    def embed_documents(self, texts):
        return self._embed(texts)

    def embed_query(self, text):
        return self._embed([text])[0]

llm

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

if OPENAI_API_KEY:
    from langchain_openai import OpenAIEmbeddings
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
else:
    embeddings = HashEmbeddings(n_features=1536)

vector_store = InMemoryVectorStore(embedding=embeddings)

In [ ]:
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building a LangChain project with retrieval examples.",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [ ]:
documents

In [ ]:
vector_store.add_documents(documents=documents)

In [ ]:
vector_store.similarity_search("hows the weather forecast")

In [ ]:
vector_store.similarity_search("hows the weather forecast",k=2)

In [ ]:
### vectorstore to retriever

retriever=vector_store.as_retriever(search_kwargs={"k":2})

retriever

In [ ]:
## Invoke
retriever.invoke("hows the weather forecast")